# 🌉 Bridge Damage Detection — Benchmark Notebook (Final)
**Dataset:** dacl10k | **Task:** Semantic Segmentation | **Classes:** 19

| Step | Deskripsi |
|---|---|
| 1 | Cek GPU + Mount Google Drive |
| 2 | Install dependencies |
| 3 | Setup: `__init__.py` + import helper |
| 4 | Verifikasi dataset |
| 5 | Konfigurasi & DataLoader |
| 6 | Load model |
| 7 | Training (auto-resume) |
| 8 | Plot training curves |
| 9 | Evaluasi test set (**fix SegFormer shape**) |
| 10 | Robustness evaluation |
| 11 | Visualisasi prediksi |
| 12 | Gabungkan hasil benchmark |

> **Sebelum mulai:** Runtime → Change runtime type → **GPU (A100 / T4)**
>
> **Fix yang sudah diintegrasikan:**
> - Auto-resume dari checkpoint (tidak perlu mulai ulang)
> - Fix SegFormer logits shape `(B,C,H/4,W/4)` → upsample ke `(B,C,H,W)`
> - Fix `torchmetrics` versi lama → `JaccardIndex(task='multiclass')`
> - Fix `GradScaler` dan `autocast` deprecated API
> - Fix `import_from_file` untuk bypass sys.path issues
> - Fix `__init__.py` otomatis dibuat

## ✅ Step 1 — Cek GPU & Mount Google Drive

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: Tidak ada GPU! Ganti runtime ke GPU dulu.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys

# ⚠️ SESUAIKAN path project di Google Drive Anda
PROJECT_PATH = '/content/drive/MyDrive/Penelitian/bridge-benchmark-cv'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    if PROJECT_PATH not in sys.path:
        sys.path.insert(0, PROJECT_PATH)
    print(f'Working directory: {os.getcwd()}')
    print('Isi folder:')
    for item in sorted(os.listdir('.')):
        print(f'  {item}')
else:
    print(f'ERROR: Path tidak ditemukan: {PROJECT_PATH}')

## 📦 Step 2 — Install Dependencies

In [ ]:
!pip install -q "transformers>=4.41.0"
!pip install -q timm segmentation-models-pytorch
!pip install -q "albumentations==1.4.3"
!pip install -q "torchmetrics>=1.3.0"
!pip install -q einops pycocotools

!pip install -q wandb

import torch, albumentations, transformers, torchmetrics
import segmentation_models_pytorch as smp
import timm
print(f'torch          : {torch.__version__}')
print(f'transformers   : {transformers.__version__}')
print(f'timm           : {timm.__version__}')
print(f'smp            : {smp.__version__}')
print(f'albumentations : {albumentations.__version__}')
print(f'torchmetrics   : {torchmetrics.__version__}')
print('\n✅ Semua dependencies OK!')

## 🔧 Step 3 — Setup: `__init__.py` + Import Helper

In [ ]:
import os, sys, importlib.util

# Buat __init__.py di semua subfolder
for folder in ['train', 'datasets', 'models', 'eval']:
    init = os.path.join(folder, '__init__.py')
    os.makedirs(folder, exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').write('')
        print(f'Created: {init}')
    else:
        print(f'OK     : {init}')

def import_from_file(module_name, file_path):
    """
    Load module langsung dari path file.
    Cara paling robust untuk import di Colab — bypass sys.path issues.
    """
    if module_name in sys.modules:
        del sys.modules[module_name]
    spec   = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

print('\nimport_from_file helper siap.')

## 🔑 Step 3b — WandB Login

> API key dibaca otomatis dari `wandb-key.cfg` di folder project.
> File ini di-gitignore — tidak akan ter-upload ke GitHub.

In [ ]:
import wandb
from pathlib import Path

# Baca API key dari wandb-key.cfg
_cfg = Path('wandb-key.cfg')
if _cfg.exists():
    _lines = _cfg.read_text().strip().splitlines()
    _api_key = _lines[1].strip()   # baris ke-2 = API key
    wandb.login(key=_api_key)
    print(f'WandB login OK')
    print(f'Project : bridge-benchmark-cv')
    print(f'Key ID  : {_lines[4].split(": ")[1] if len(_lines) > 4 else "?"}')
else:
    print('WARNING: wandb-key.cfg tidak ditemukan!')
    print('Jalankan: wandb login (manual)')
    wandb.login()


## 🗂️ Step 4 — Verifikasi Dataset

In [ ]:
from pathlib import Path

DATA_ROOT = 'data/dacl10k'
print('=== Verifikasi Struktur Dataset ===')
total = 0
for split in ['train', 'val', 'test']:
    img_dir = Path(DATA_ROOT) / split / 'img'
    ann_dir = Path(DATA_ROOT) / split / 'ann'
    imgs = list(img_dir.glob('*.*')) if img_dir.exists() else []
    anns = list(ann_dir.glob('*.json')) if ann_dir.exists() else []
    total += len(imgs)
    status = 'OK' if len(imgs) > 0 else 'MISSING'
    print(f'  [{status}] [{split:5s}] {len(imgs):5d} gambar | {len(anns):5d} JSON')
print(f'\nTotal: {total} gambar')
if total == 0:
    print('ERROR: Dataset kosong! Pastikan dacl10k sudah di-download.')

In [ ]:
# Load dacl10k_loader dan test satu sampel
loader_mod = import_from_file(
    'datasets.dacl10k_loader',
    'datasets/dacl10k_loader.py'
)
Dacl10kDataset   = loader_mod.Dacl10kDataset
build_dataloaders = loader_mod.build_dataloaders
build_robust_loader = loader_mod.build_robust_loader
ALL_CLASSES      = loader_mod.ALL_CLASSES
NUM_CLASSES      = loader_mod.NUM_CLASSES

ds = Dacl10kDataset(root_dir=DATA_ROOT, split='train', debug=True)
s  = ds[0]
print(f'Sample [0]:')
print(f'  image  : {s["image"].shape}  {s["image"].dtype}')
print(f'  mask   : {s["mask"].shape}   {s["mask"].dtype}')
print(f'  labels : {s["mask"].unique().tolist()}')
print(f'\n✅ Dataset OK! ({NUM_CLASSES} kelas)')

## ⚙️ Step 5 — Konfigurasi & DataLoader

> Ganti `MODEL_NAME` untuk training model berbeda:
> `'segformer'` · `'deeplabv3'` · `'unet_effnet'` · `'unet_resnet'`

In [ ]:
import torch

# ── GANTI DI SINI UNTUK MODEL BERBEDA ──────────────────────────
MODEL_NAME  = 'deeplabv3'
# ────────────────────────────────────────────────────────────────

DATA_ROOT   = 'data/dacl10k'
IMG_SIZE    = 512
BATCH_SIZE  = 8       # Kurangi ke 4 jika OOM
NUM_WORKERS = 2
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

CONFIG = {
    'model_name'    : MODEL_NAME,
    'epochs'        : 50,
    'lr'            : 1e-4,
    'weight_decay'  : 1e-4,
    'loss_alpha'    : 0.5,
    'use_focal'     : False,
    'scheduler'     : 'cosine',
    'warmup_epochs' : 3,
    'patience'      : 10,
    'use_amp'       : True,
    'num_classes'   : 19,
    'ignore_index'  : 255,
    'img_size'      : IMG_SIZE,
    'output_dir'    : 'outputs',
    # ── WandB ──────────────────────────────
    'use_wandb'     : True,
    'wandb_project' : 'bridge-benchmark-cv',
    'wandb_entity'  : None,   # auto-detect dari API key
}

train_loader, val_loader, test_loader = build_dataloaders(
    root_dir    = DATA_ROOT,
    img_size    = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
)
print(f'Model    : {MODEL_NAME}')
print(f'Device   : {DEVICE}')
print(f'Train    : {len(train_loader)} batches')
print(f'Val      : {len(val_loader)} batches')
print(f'Test     : {len(test_loader)} batches')
print('\n✅ DataLoaders siap!')

## 🤖 Step 6 — Load Model

In [ ]:
def load_model(model_name, num_classes=19):
    """Factory function — load model berdasarkan nama."""
    import segmentation_models_pytorch as smp
    from transformers import SegformerForSemanticSegmentation

    if model_name == 'segformer':
        print('Loading SegFormer-B2 (Transformer)...')
        model = SegformerForSemanticSegmentation.from_pretrained(
            'nvidia/segformer-b2-finetuned-ade-512-512',
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
        )
    elif model_name == 'deeplabv3':
        print('Loading DeepLabV3+ (ResNet50)...')
        model = smp.DeepLabV3Plus(
            encoder_name='resnet50', encoder_weights='imagenet',
            in_channels=3, classes=num_classes)
    elif model_name == 'unet_effnet':
        print('Loading UNet-EfficientNetB4...')
        model = smp.Unet(
            encoder_name='efficientnet-b4', encoder_weights='imagenet',
            in_channels=3, classes=num_classes)
    elif model_name == 'unet_resnet':
        print('Loading UNet-ResNet101...')
        model = smp.Unet(
            encoder_name='resnet101', encoder_weights='imagenet',
            in_channels=3, classes=num_classes)
    else:
        raise ValueError(f'Model tidak dikenal: {model_name}')

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Parameters: {n_params:.1f}M')
    return model

model = load_model(MODEL_NAME, num_classes=19)
print('\n✅ Model loaded!')

## 🏋️ Step 7 — Training (Auto-Resume)

> Cell ini **otomatis** mendeteksi checkpoint:
> - Ada checkpoint → lanjut dari epoch terakhir
> - Tidak ada → mulai dari awal

In [ ]:
import os

# Reload modules dengan fix terbaru
losses_mod  = import_from_file('train.losses',  'train/losses.py')
trainer_mod = import_from_file('train.trainer', 'train/trainer.py')
Trainer     = trainer_mod.Trainer
print('✅ Trainer imported!')

os.makedirs(f'outputs/{MODEL_NAME}', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

LAST_CKPT = f'outputs/{MODEL_NAME}/last_checkpoint.pth'

trainer = Trainer(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    config       = CONFIG,
    device       = DEVICE,
)

if os.path.exists(LAST_CKPT):
    ckpt       = torch.load(LAST_CKPT, map_location=DEVICE)
    last_epoch = ckpt['epoch']
    print(f'\nCheckpoint ditemukan!')
    print(f'  Epoch terakhir : {last_epoch}')
    print(f'  Best mIoU      : {ckpt["best_miou"]:.4f}')
    start_epoch = trainer.load_checkpoint(LAST_CKPT)
    for _ in range(last_epoch):
        trainer.scheduler.step()
    print(f'  Scheduler fast-forward ke epoch {last_epoch} OK')
    print(f'\nMelanjutkan dari epoch {start_epoch}...')
else:
    start_epoch = 1
    print('\nTidak ada checkpoint — training dari awal...')

best_miou = trainer.fit(start_epoch=start_epoch)
print(f'\n🏆 Training selesai! Best val mIoU = {best_miou:.4f}')

## 📈 Step 8 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'outputs/{MODEL_NAME}/training_log.csv'
if not os.path.exists(log_path):
    print(f'Log tidak ditemukan: {log_path}')
else:
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Training Curves — {MODEL_NAME}', fontsize=13, fontweight='bold')

    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='#3b82f6')
    axes[0].plot(df['epoch'], df['val_loss'],   label='Val Loss',   color='#ef4444', ls='--')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(df['epoch'], df['val_miou'], color='#10b981', lw=2, label='Val mIoU')
    best_ep = df['val_miou'].idxmax()
    axes[1].axvline(df['epoch'][best_ep], color='orange', ls=':', lw=1.5,
                   label=f'Best epoch {df["epoch"][best_ep]} ({df["val_miou"][best_ep]:.4f})')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
    axes[1].set_title('Validation mIoU'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'outputs/figures/{MODEL_NAME}_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Best mIoU: {df["val_miou"].max():.4f} @ epoch {df["epoch"][best_ep]}')

## 📊 Step 9 — Evaluasi Test Set

> **Fix diintegrasikan:**
> - SegFormer output `(B,C,H/4,W/4)` di-upsample ke `(B,C,H,W)` sebelum argmax
> - `JaccardIndex(task='multiclass')` untuk torchmetrics versi baru
> - Semua model menggunakan pipeline evaluasi yang sama

In [ ]:
import torch
import time
from torchmetrics import JaccardIndex
from torchmetrics.classification import MulticlassF1Score
from tqdm.notebook import tqdm

def extract_logits(outputs, target_size):
    """
    Ekstrak logits dari berbagai format output model.
    Fix untuk SegFormer yang mengembalikan dict/object dengan .logits
    dan output berukuran H/4 x W/4.
    """
    # Ekstrak logits dari berbagai format
    if hasattr(outputs, 'logits'):        # HuggingFace SegFormer
        logits = outputs.logits
    elif isinstance(outputs, dict):
        logits = outputs.get('logits', outputs.get('out'))
    else:
        logits = outputs

    # Upsample jika ukuran berbeda (WAJIB untuk SegFormer)
    if logits.shape[-2:] != target_size:
        logits = torch.nn.functional.interpolate(
            logits,
            size=target_size,
            mode='bilinear',
            align_corners=False
        )
    return logits

# Load best model
ckpt = torch.load(
    f'outputs/{MODEL_NAME}/best_model.pth',
    map_location=DEVICE
)
model.load_state_dict(ckpt['model_state'])
model = model.to(DEVICE)
model.eval()
print(f'Loaded: epoch={ckpt["epoch"]} | training mIoU={ckpt["best_miou"]:.4f}')

# Cek output shape pada 1 batch pertama
with torch.no_grad():
    sample_batch = next(iter(test_loader))
    sample_out   = model(sample_batch['image'].to(DEVICE))
    raw_logits   = sample_out.logits if hasattr(sample_out, 'logits') else sample_out
    fixed_logits = extract_logits(sample_out, (IMG_SIZE, IMG_SIZE))
print(f'\nOutput shape check:')
print(f'  Raw logits   : {raw_logits.shape}')
print(f'  Fixed logits : {fixed_logits.shape}')
print(f'  Target mask  : {sample_batch["mask"].shape}')
assert fixed_logits.shape[-2:] == (IMG_SIZE, IMG_SIZE), 'Shape mismatch!'
print('  Shape OK!')

In [ ]:
# Evaluasi penuh
metric_iou = JaccardIndex(
    task='multiclass', num_classes=NUM_CLASSES,
    average='none', ignore_index=255
).to(DEVICE)

metric_f1 = MulticlassF1Score(
    num_classes=NUM_CLASSES,
    average='none',
    ignore_index=255
).to(DEVICE)

# FPS measurement
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    for _ in range(10): model(dummy)   # warmup
    t0 = time.time()
    for _ in range(50): model(dummy)
    fps = 50 / (time.time() - t0)
n_params = sum(p.numel() for p in model.parameters()) / 1e6

# Evaluasi mIoU & F1
with torch.no_grad():
    for batch in tqdm(test_loader, desc=f'Evaluasi {MODEL_NAME}'):
        images  = batch['image'].to(DEVICE)
        targets = batch['mask'].to(DEVICE)
        outputs = model(images)
        logits  = extract_logits(outputs, targets.shape[-2:])
        preds   = logits.argmax(1)
        metric_iou.update(preds, targets)
        metric_f1.update(preds, targets)

miou_per_class = metric_iou.compute().cpu()
f1_per_class   = metric_f1.compute().cpu()
miou_mean      = miou_per_class.mean().item()
f1_mean        = f1_per_class.mean().item()

print(f'\n=== Evaluasi: {MODEL_NAME} ===')
print(f'{"Idx":<4} {"Class":<32} {"IoU":>8} {"F1":>8}')
print('-' * 56)
for i, (cls, iou, f1) in enumerate(zip(ALL_CLASSES, miou_per_class, f1_per_class)):
    bar = '█' * int(iou.item() * 20)
    print(f'[{i:2d}] {cls:<32} {iou.item():>6.4f}  {f1.item():>6.4f}  {bar}')
print('-' * 56)
print(f'     {"MEAN":<32} {miou_mean:>6.4f}  {f1_mean:>6.4f}')
print(f'\nFPS    : {fps:.1f}')
print(f'Params : {n_params:.1f}M')

In [ ]:
import pandas as pd, os
os.makedirs('outputs/results', exist_ok=True)

results_df = pd.DataFrame({
    'class' : ALL_CLASSES + ['MEAN'],
    'iou'   : miou_per_class.tolist() + [miou_mean],
    'f1'    : f1_per_class.tolist() + [f1_mean],
    'model' : MODEL_NAME,
    'fps'   : fps,
    'params_m' : n_params,
})
csv_path = f'outputs/results/{MODEL_NAME}_test_results.csv'
results_df.to_csv(csv_path, index=False)
print(f'Tersimpan: {csv_path}')
print(f'\n🏆 {MODEL_NAME}: mIoU={miou_mean:.4f} | F1={f1_mean:.4f} | FPS={fps:.1f} | Params={n_params:.1f}M')

# ── WandB: log test results ─────────────────────────────────────────
if wandb.run is not None:
    _tbl = wandb.Table(columns=["class", "IoU", "F1"])
    for _cls, _iou, _f1 in zip(ALL_CLASSES, miou_per_class, f1_per_class):
        _tbl.add_data(_cls, round(_iou.item(), 4), round(_f1.item(), 4))

    _log = {
        "test/mIoU"        : miou_mean,
        "test/mF1"         : f1_mean,
        "test/fps"         : fps,
        "test/params_M"    : n_params,
        "test/per_class"   : _tbl,
    }
    for _cls, _iou in zip(ALL_CLASSES, miou_per_class):
        _log[f"test/iou/{_cls}"] = _iou.item()

    wandb.log(_log)
    print("\n[WandB] test results logged")


## 🌫️ Step 10 — Robustness Evaluation

In [ ]:
robust_results = {'normal': miou_mean}

for corruption in ['blur', 'noise', 'brightness']:
    rl = build_robust_loader(
        root_dir=DATA_ROOT, corruption=corruption,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
    )
    m = JaccardIndex(
        task='multiclass', num_classes=NUM_CLASSES,
        average='none', ignore_index=255
    ).to(DEVICE)

    with torch.no_grad():
        for batch in tqdm(rl, desc=f'Robust [{corruption}]'):
            images  = batch['image'].to(DEVICE)
            targets = batch['mask'].to(DEVICE)
            outputs = model(images)
            logits  = extract_logits(outputs, targets.shape[-2:])
            m.update(logits.argmax(1), targets)

    rob_miou = m.compute().mean().item()
    delta    = miou_mean - rob_miou
    robust_results[corruption] = rob_miou
    print(f'  [{corruption:<12}] mIoU={rob_miou:.4f}  delta={delta:+.4f}')

print(f'\n=== Robustness: {MODEL_NAME} ===')
for k, v in robust_results.items():
    delta = (miou_mean - v) if k != 'normal' else 0.0
    print(f'  {k:<12} : {v:.4f}  (delta={delta:+.4f})')

# Simpan robustness CSV
import pandas as pd
rob_df = pd.DataFrame([{
    'model'           : MODEL_NAME,
    'normal'          : robust_results.get('normal', 0),
    'blur'            : robust_results.get('blur', 0),
    'noise'           : robust_results.get('noise', 0),
    'brightness'      : robust_results.get('brightness', 0),
    'delta_blur'      : miou_mean - robust_results.get('blur', miou_mean),
    'delta_noise'     : miou_mean - robust_results.get('noise', miou_mean),
    'delta_brightness': miou_mean - robust_results.get('brightness', miou_mean),
}])
rob_df.to_csv(f'outputs/results/{MODEL_NAME}_robustness.csv', index=False)
print(f'\nTersimpan: outputs/results/{MODEL_NAME}_robustness.csv')

# ── WandB: log robustness results ───────────────────────────────────
if wandb.run is not None:
    _rob_log = {}
    for _k, _v in robust_results.items():
        if _k == "normal":
            continue
        _rob_log[f"robustness/{_k}"]       = _v
        _rob_log[f"robustness/delta_{_k}"] = miou_mean - _v
    wandb.log(_rob_log)
    print("[WandB] robustness results logged")


## 🖼️ Step 11 — Visualisasi Prediksi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CLASS_COLORS = [
    (255,192,203),(178,223,138),(0,255,128),(0,0,255),(139,0,0),
    (255,140,0),(169,169,169),(207,52,118),(128,0,0),(246,124,104),
    (128,0,128),(128,128,0),(212,175,162),(253,191,111),(255,0,0),
    (123,211,247),(0,0,139),(0,128,0),(0,139,139),
]
def denormalize(t):
    mean = np.array([0.485,0.456,0.406])
    std  = np.array([0.229,0.224,0.225])
    img  = t.permute(1,2,0).numpy()
    return (img*std+mean).clip(0,1)

def mask_to_rgb(m):
    mask = m.numpy()
    rgb  = np.zeros((*mask.shape,3), dtype=np.uint8)
    for idx, c in enumerate(CLASS_COLORS):
        rgb[mask==idx] = c
    rgb[mask==255] = (50,50,50)
    return rgb

model.eval()
batch   = next(iter(test_loader))
images  = batch['image'][:4].to(DEVICE)
targets = batch['mask'][:4]

with torch.no_grad():
    outputs = model(images)
    logits  = extract_logits(outputs, targets.shape[-2:])
    preds   = logits.argmax(1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle(f'Prediksi Segmentasi — {MODEL_NAME}', fontsize=13, fontweight='bold')

for col in range(4):
    axes[0,col].imshow(denormalize(batch['image'][col]))
    axes[0,col].set_title(f'Input [{col}]', fontsize=8)
    axes[1,col].imshow(mask_to_rgb(targets[col]))
    axes[2,col].imshow(mask_to_rgb(preds[col]))
    for row in range(3): axes[row,col].axis('off')

for row, label in enumerate(['Input Image','Ground Truth','Prediction']):
    axes[row,0].set_ylabel(label, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'outputs/figures/{MODEL_NAME}_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Tersimpan: outputs/figures/{MODEL_NAME}_predictions.png')

# ── WandB: log prediction figure ─────────────────────────────────────
if wandb.run is not None:
    wandb.log({"test/predictions_figure":
               wandb.Image(f'outputs/figures/{MODEL_NAME}_predictions.png')})
    print("[WandB] prediction figure logged")


## 📋 Step 12 — Gabungkan Semua Hasil Benchmark

> Jalankan setelah **semua model selesai**: segformer, deeplabv3, unet_effnet, unet_resnet.
> (YOLOv8 dan YOLOv11 ada di notebook terpisah)

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

results_dir = Path('outputs/results')
csvs = sorted(results_dir.glob('*_test_results.csv'))

if len(csvs) == 0:
    print('Belum ada hasil. Jalankan evaluasi model dulu.')
else:
    print(f'Ditemukan {len(csvs)} file hasil:')
    for c in csvs: print(f'  {c.name}')

    dfs = [pd.read_csv(c) for c in csvs]
    combined = pd.concat(dfs)
    pivot = combined.pivot(index='class', columns='model', values='iou')
    mean_row   = pivot.loc[['MEAN']]
    class_rows = pivot.drop('MEAN').sort_index()
    final      = pd.concat([class_rows, mean_row])

    print('\n=== TABEL BENCHMARK (mIoU) ===')
    print(final.round(4).to_string())
    final.to_csv('outputs/results/benchmark_complete.csv')
    print('\nTersimpan: outputs/results/benchmark_complete.csv')

    # Bar chart (Fig. 5 paper)
    mean_vals = mean_row.iloc[0].sort_values(ascending=False)
    color_map = {
        'segformer':'#8b5cf6','swinunet':'#a78bfa',
        'yolov8':'#3b82f6','yolov11':'#60a5fa',
        'deeplabv3':'#10b981','unet_effnet':'#34d399','unet_resnet':'#6ee7b7',
    }
    colors = [color_map.get(m,'#94a3b8') for m in mean_vals.index]

    fig, ax = plt.subplots(figsize=(12,5))
    bars = ax.bar(mean_vals.index, mean_vals.values, color=colors, edgecolor='white')
    ax.axhline(y=0.42, color='red', ls='--', lw=1.5, label='Baseline dacl10k (0.42)')
    ax.set_ylabel('Mean IoU', fontsize=11)
    ax.set_title('Benchmark mIoU — dacl10k', fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(mean_vals.values)*1.15)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, mean_vals.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/figures/benchmark_miou_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Tersimpan: outputs/figures/benchmark_miou_comparison.png')

# ── WandB: tutup run ─────────────────────────────────────────────────
if wandb.run is not None:
    wandb.finish()
    print("[WandB] Run selesai — semua metrics terecord.")


In [ ]:
# Checklist status semua model
print('=== CHECKLIST MODEL ===')
checklist = ['segformer','deeplabv3','unet_effnet','unet_resnet','yolov8','yolov11']
for model_name in checklist:
    ckpt_path = f'outputs/{model_name}/best_model.pth'
    csv_path  = f'outputs/results/{model_name}_test_results.csv'
    has_ckpt  = Path(ckpt_path).exists()
    has_csv   = Path(csv_path).exists()

    if has_csv:
        df = pd.read_csv(csv_path)
        miou = df[df['class']=='MEAN']['iou'].values[0]
        status = f'DONE  mIoU={miou:.4f}'
    elif has_ckpt:
        import torch
        ck = torch.load(ckpt_path, map_location='cpu')
        status = f'Training done (epoch {ck["epoch"]}) — belum dievaluasi'
    else:
        status = 'Belum dimulai'

    print(f'  [{model_name:<15}] {status}')